# 10 · Capstone: a multi-agent research assistant (+ going to production)

**Course:** Building LLM Agents, notebook 10 of 10
**Where we are:** the finale. Nine notebooks of building blocks come together here. We assemble a multi-agent research assistant end to end, then cover what it takes to run agents in production.

### What you'll learn
- How to wire the whole course together: tools, the agent loop, memory, robustness, multiple agents, orchestration, and reflection, in one working system
- A clean `run_research(question)` pipeline: Planner, parallel Researchers, Writer, Critic, one revision
- Going to production: cost and token tracking, observability/tracing, reliability (retries, validation, fallbacks), guardrails and safety, evaluation, and when not to use agents at all
- Where to go next: the real frameworks (LangGraph, CrewAI, AutoGen, OpenAI Agents SDK, LlamaIndex) and what each is good for

### Prerequisites
- You've worked through notebooks 01 to 09 (we reuse ideas from every one)
- Your `.env` has a working Sigma2 token

> Mental model: a multi-agent system is the same one ability from notebook 01 (predict the next token) wrapped in roles and control flow you design. The Planner, Researchers, Writer, and Critic are each an LLM call (or an agent loop) with a focused job. Your job as the engineer is the orchestration between them, and, in production, everything around them: cost, logging, retries, guardrails, and evaluation.

> Recap of the journey so far: notebook 03 gave us tools; 04 put a model in a loop (reason, act, observe), the `Agent`/`Tool` classes we import below; 05 added memory (buffers, summarisation, lexical retrieval); 06 made a single agent robust (error handling, guardrails); 07 introduced multiple specialised agents passing messages; 08 covered orchestration patterns (sequential pipeline, router, orchestrator-worker, parallel fan-out); and 09 added planning and reflection (planner-executor, self-critique). This notebook stitches them into one system.

## 0. Setup

The familiar one-liner. We import the ready client, the model id, the reusable `Agent`/`Tool` classes (the ones you built by hand in notebook 04), and `show_messages` for debugging.

In [ ]:
from helpers import client, MODEL, Agent, Tool, ask, show_messages

import json
import time
from concurrent.futures import ThreadPoolExecutor

print("Model:", MODEL)

# Part A: Build the system

We're building a research/report assistant. You give it a question; it produces a short structured report. The pipeline has four roles, each one a thing you already know how to build:

| Role | What it does | Notebook it draws on |
|---|---|---|
| Planner | breaks the question into 2 to 3 subtopics (returns JSON) | 02 (JSON), 09 (planning) |
| Researcher ×N | gathers notes for one subtopic, using a search tool | 03 (tools), 04 (loop), runs in parallel (08) |
| Writer | synthesises all the notes into a report | 02, 07 (roles) |
| Critic | reviews the report and triggers one revision | 09 (reflection) |

> The shape: `Planner` (1 call), `Researchers` (N calls, in parallel), `Writer` (1 call), `Critic` (1 call), then maybe one `Writer` revision. That combines the orchestrator-worker and reflection patterns from 08 and 09.

![The full capstone architecture, annotated with each pattern used](figures/10-capstone-arch.svg)

*Figure: Planner, parallel Researchers, Writer, Critic (one revision), the whole system. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## A1. A self-contained knowledge tool

Researchers need something to search. In production this would be a real web-search API or a RAG retriever over your documents (notebook 05 showed lexical retrieval; production would add a vector store, and recall that embeddings aren't available on our endpoint, so we'd add that separately).

For a notebook that runs anywhere, offline, we fake it: a small in-memory "knowledge base" and a `search_kb` tool that does plain keyword matching over it. The interface is exactly what a real search tool would expose, so swapping in the real thing later is a drop-in change.

In [ ]:
# A tiny offline "knowledge base": topic -> list of canned facts.
# In production this is a search API or a RAG retriever. The TOOL interface
# (a query string in, matching snippets out) stays identical.
KNOWLEDGE_BASE = {
    "solar": [
        "Solar PV costs fell roughly 90% between 2010 and 2023, making it the cheapest new electricity in many regions.",
        "Solar is intermittent: it generates only when the sun shines, so it pairs with storage or flexible demand.",
    ],
    "wind": [
        "Offshore wind has higher capacity factors than onshore but costs more to build and maintain.",
        "Wind and solar are complementary: wind often peaks at night and in winter when solar is weak.",
    ],
    "storage": [
        "Lithium-ion battery storage smooths short-term (hours) variability; costs have fallen sharply since 2015.",
        "For seasonal storage, options like pumped hydro and green hydrogen are discussed but remain costly.",
    ],
    "grid": [
        "High shares of renewables require grid upgrades and interconnection to move power across regions.",
        "Demand response and smart grids help match flexible consumption to variable generation.",
    ],
    "cost": [
        "The 'levelised cost of energy' (LCOE) lets you compare technologies on a per-MWh basis.",
        "Renewables now often beat fossil fuels on LCOE, but system integration adds extra cost.",
    ],
}


def search_kb(query: str) -> str:
    """Search the knowledge base for snippets relevant to the query.

    Returns up to 4 matching facts as a newline-separated string, or a
    'no results' message. (A stand-in for a real web-search / RAG tool.)
    """
    q = query.lower()
    words = [w for w in q.split() if len(w) > 3]
    hits = []
    for topic, facts in KNOWLEDGE_BASE.items():
        if topic in q:
            hits.extend(facts)            # topic name matches the query
        else:
            # also match on any overlapping word, so free-text queries work
            for fact in facts:
                if any(w in fact.lower() for w in words):
                    hits.append(fact)
    hits = list(dict.fromkeys(hits))[:4]  # dedupe, cap at 4
    return "\n".join(f"- {h}" for h in hits) if hits else "No results found."


# quick sanity check
print(search_kb("how cheap is solar power"))

## A2. The Planner

The Planner takes the question and returns 2 to 3 subtopics as JSON. We use json mode (notebook 02) and `temperature=0`: this is a decision, not creative writing, so we want it stable.

> Key idea: decomposition is the highest-leverage step. Good subtopics make the researchers' job easy; vague ones make everything downstream worse. This is the planner-executor pattern from notebook 09.

In [ ]:
def plan(question: str, n: int = 3) -> list[str]:
    """Decompose a question into a small list of focused subtopics."""
    system = (
        "You are a research planner. Break the user's question into "
        f"{n} focused, non-overlapping subtopics that together answer it. "
        'Reply ONLY with JSON: {"subtopics": ["...", "..."]}'
    )
    raw = ask(
        question,
        system=system,
        temperature=0,
        response_format={"type": "json_object"},
        max_tokens=8000,
    )
    data = json.loads(raw)
    subs = data.get("subtopics", [])[:n]
    return subs


# try it
demo_subs = plan("How can a country run on mostly renewable energy?", n=2)
demo_subs

## A3. The Researcher (a tool-using agent)

Each Researcher is a small agent (notebook 04) with one tool: `search_kb`. It looks up its subtopic and returns a few tight notes. Because each researcher is independent, we can run them in parallel, which we'll do next.

We give it a low `max_steps`: a researcher should search once or twice and stop, not wander.

In [ ]:
search_tool = Tool(
    search_kb,
    description="Search the knowledge base for facts relevant to a query.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "what to look up"}
        },
        "required": ["query"],
    },
)


def research(subtopic: str, verbose: bool = False) -> str:
    """Run a researcher agent on one subtopic; return its notes as text."""
    agent = Agent(
        name=f"researcher",
        system=(
            "You are a researcher. Use the search tool to gather facts about "
            "the given subtopic, then summarise them as 2-3 concise bullet "
            "points. Only state facts you found via the tool."
        ),
        tools=[search_tool],
        max_steps=4,
        temperature=0,
    )
    return agent.run(f"Research this subtopic: {subtopic}", verbose=verbose)


# one researcher, verbose, so you can see the tool round-trip
print(research(demo_subs[0], verbose=True))

## A4. Running the researchers in parallel

The researchers don't depend on each other, so running them one by one just wastes wall-clock time. We reuse the parallel fan-out pattern from notebook 08: a `ThreadPoolExecutor` fires all the LLM calls concurrently (network-bound work, so threads are fine).

In [ ]:
def research_all(subtopics: list[str], verbose: bool = False) -> dict[str, str]:
    """Run one researcher per subtopic, in parallel. Returns {subtopic: notes}."""
    with ThreadPoolExecutor(max_workers=len(subtopics)) as pool:
        notes = list(pool.map(lambda s: research(s, verbose=verbose), subtopics))
    return dict(zip(subtopics, notes))


t0 = time.time()
demo_notes = research_all(demo_subs)
print(f"researched {len(demo_notes)} subtopics in {time.time() - t0:.1f}s (in parallel)\n")
for sub, note in demo_notes.items():
    print(f"### {sub}\n{note}\n")

## A5. The Writer

The Writer synthesises all the notes into a short structured report. This is generation, so we allow a little temperature, but keep it modest for a factual report.

In [ ]:
def write_report(question: str, notes: dict[str, str]) -> str:
    """Synthesise per-subtopic notes into one structured report."""
    body = "\n\n".join(f"## {sub}\n{note}" for sub, note in notes.items())
    system = (
        "You are a technical writer. Using ONLY the supplied notes, write a "
        "short report (a one-paragraph summary, then a section per subtopic). "
        "Do not invent facts beyond the notes."
    )
    prompt = f"Question: {question}\n\nNotes:\n{body}\n\nWrite the report."
    return ask(prompt, system=system, temperature=0.3, max_tokens=8000)


demo_report = write_report("How can a country run on mostly renewable energy?", demo_notes)
print(demo_report)

## A6. The Critic (reflection + one revision)

The Critic reads the report and decides whether it's good enough. This is the reflection loop from notebook 09, but bounded to exactly one revision so it can't loop forever (and burn tokens). The Critic returns JSON: a verdict plus, if needed, concrete feedback.

> Gotcha: reflection without a hard cap is a classic way to set money on fire. Always bound it: "at most one revision", "at most N critique rounds". We'll see the cost of this run at the end.

In [ ]:
def critique(question: str, report: str) -> dict:
    """Review a report. Returns {'ok': bool, 'feedback': str}."""
    system = (
        "You are a strict editor. Judge whether the report clearly and "
        "accurately answers the question using its notes. "
        'Reply ONLY with JSON: {"ok": true/false, "feedback": "one sentence of '
        'specific, actionable feedback (empty if ok)"}'
    )
    prompt = f"Question: {question}\n\nReport:\n{report}"
    raw = ask(
        prompt, system=system, temperature=0,
        response_format={"type": "json_object"}, max_tokens=8000,
    )
    data = json.loads(raw)
    return {"ok": bool(data.get("ok")), "feedback": data.get("feedback", "")}


def revise(question: str, report: str, feedback: str, notes: dict[str, str]) -> str:
    """Rewrite the report once, addressing the critic's feedback."""
    body = "\n\n".join(f"## {sub}\n{note}" for sub, note in notes.items())
    system = (
        "You are a technical writer revising a report. Apply the editor's "
        "feedback. Use ONLY the supplied notes; do not invent facts."
    )
    prompt = (
        f"Question: {question}\n\nNotes:\n{body}\n\n"
        f"Current report:\n{report}\n\nEditor feedback: {feedback}\n\n"
        "Rewrite the report addressing the feedback."
    )
    return ask(prompt, system=system, temperature=0.3, max_tokens=8000)


demo_review = critique("How can a country run on mostly renewable energy?", demo_report)
demo_review

## A7. Orchestrating it all: `run_research`

Now we glue the four roles into one readable pipeline. We thread a tracer through every step (a list of log records): that's our observability hook for Part B, and it doubles as a live progress printout.

Notice the shape of the function: it reads top to bottom like the diagram. That is the point of orchestration: the control flow lives in plain Python you can read, not hidden inside a prompt.

In [ ]:
def run_research(question: str, n_subtopics: int = 2, verbose: bool = True) -> dict:
    """End-to-end research assistant: plan -> research (parallel) -> write -> critique -> maybe revise."""
    trace = []

    def step(name, detail=""):
        trace.append({"t": round(time.time() - start, 2), "step": name, "detail": detail})
        if verbose:
            print(f"[{trace[-1]['t']:>5}s] {name}: {detail}")

    start = time.time()
    step("plan", "decomposing question")
    subtopics = plan(question, n=n_subtopics)
    step("plan.done", f"{len(subtopics)} subtopics: {subtopics}")

    step("research", f"dispatching {len(subtopics)} researchers in parallel")
    notes = research_all(subtopics)
    step("research.done", f"gathered notes for {len(notes)} subtopics")

    step("write", "synthesising report")
    report = write_report(question, notes)
    step("write.done", f"{len(report)} chars")

    step("critique", "reviewing report")
    review = critique(question, report)
    step("critique.done", f"ok={review['ok']} feedback={review['feedback']!r}")

    if not review["ok"] and review["feedback"]:
        step("revise", "applying critic feedback (one pass)")
        report = revise(question, report, review["feedback"], notes)
        step("revise.done", f"{len(report)} chars")
    else:
        step("revise.skip", "report accepted as-is")

    return {"question": question, "subtopics": subtopics, "notes": notes,
            "report": report, "review": review, "trace": trace}

Let's run the whole thing on one question. (We keep it to 2 subtopics and short generations to limit tokens; bump these up for real use.)

In [ ]:
result = run_research("How can a country run mostly on renewable energy?", n_subtopics=2)

print("\n" + "=" * 70 + "\nFINAL REPORT\n" + "=" * 70)
print(result["report"])

# Part B: Going to production

A pipeline that runs on your laptop is the easy 20%. Running agents reliably, affordably, and safely for real users is the other 80%. Here are the six things you will actually have to deal with.

## B1. Cost & token tracking

Back in notebook 01 we flagged the agent cost trap: a single chat call is cheap, but an agent loops and resends the whole growing history, so tokens balloon. A multi-agent system multiplies that across roles. You must measure it.

The cleanest way is to wrap the client so every `chat.completions.create` call adds its `usage` to a running total. Then any code that uses the wrapped client is tracked automatically, including our `Agent` instances, since they accept a `client=`.

In [ ]:
class TrackedClient:
    """Thin wrapper that totals token usage across every call it proxies."""

    def __init__(self, inner):
        self._inner = inner
        self.reset()
        # expose the same .chat.completions.create surface
        self.chat = type("C", (), {"completions": self})()

    def reset(self):
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0

    def create(self, *args, **kwargs):
        resp = self._inner.chat.completions.create(*args, **kwargs)
        u = getattr(resp, "usage", None)
        if u:
            self.calls += 1
            self.prompt_tokens += u.prompt_tokens
            self.completion_tokens += u.completion_tokens
        return resp

    @property
    def total_tokens(self):
        return self.prompt_tokens + self.completion_tokens

    def summary(self):
        # Illustrative price; on a shared cluster "cost" is really your token budget.
        price_per_1k = 0.0006  # pretend $/1k tokens — swap for your provider's rate
        est = self.total_tokens / 1000 * price_per_1k
        return (
            f"calls={self.calls}  prompt={self.prompt_tokens}  "
            f"completion={self.completion_tokens}  total={self.total_tokens}  "
            f"~cost=${est:.4f}"
        )


tracked = TrackedClient(client)
print("tracker ready:", tracked.summary())

To track the whole pipeline, we point our helpers at the tracked client. `ask()` uses the module-level `client`, and `Agent` takes a `client=` argument, so we make small tracked variants and re-run. (In a real codebase you'd inject the client everywhere from the start.)

In [ ]:
import helpers

# Run the pipeline again, but with usage tracked. We temporarily swap the
# client the helpers use, then restore it.
tracked.reset()
_orig_client = helpers.client
helpers.client = tracked          # ask() reads helpers.client
client_backup = globals()["client"]
try:
    globals()["client"] = tracked  # Agent() defaults to the global client
    tracked_result = run_research("What makes battery storage important for renewables?",
                                  n_subtopics=2, verbose=False)
finally:
    helpers.client = _orig_client
    globals()["client"] = client_backup

print("PER-RUN TOKEN SUMMARY")
print(tracked.summary())

> What to do with this: log the per-run total to your metrics system, set a budget cap (abort a run that exceeds N tokens), and watch the trend. The single biggest lever is usually fewer or shorter LLM calls: tighter prompts, smaller `max_tokens`, and bounding loops (like our one-revision cap).

## B2. Observability / tracing

When a multi-agent run misbehaves, "the answer is wrong" is useless. You need to see who did what, with which inputs and outputs. We already built the hook: every `run_research` call returns a `trace`. Here it is for the run above, a debuggable timeline.

In [ ]:
print("TRACE of the tracked run:\n")
for rec in tracked_result["trace"]:
    print(f"  {rec['t']:>5}s  {rec['step']:<14} {rec['detail']}")

That's a hand-rolled tracer to show the idea. In production you'd emit structured spans, one per agent step, with timing, token counts, inputs and outputs, to a tracing tool. Purpose-built ones for LLM apps include LangSmith, Langfuse, Arize Phoenix, or generic OpenTelemetry. The non-negotiable: every model call and tool call is logged with enough context to replay it.

> Rule of thumb: if you can't reconstruct exactly what each agent saw and said from your logs, you can't debug it. Build tracing in from day one, not after the first incident.

## B3. Reliability: retries, timeouts, validation, fallback

Networks blip and models occasionally return junk. Three habits make agents reliable:

1. Retry with backoff on transient API errors (and set a timeout so a hung call can't stall the pipeline).
2. Validate model output before trusting it, especially JSON.
3. Fail gracefully: if a step can't recover, fall back to a safe default instead of crashing the whole run.

In [ ]:
def with_retries(fn, *, attempts=3, base_delay=0.5, fallback=None):
    """Call fn() with exponential backoff; return fallback if all attempts fail."""
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            if i == attempts:
                print(f"  [reliability] all {attempts} attempts failed ({e}); using fallback")
                return fallback
            wait = base_delay * (2 ** (i - 1))
            print(f"  [reliability] attempt {i} failed ({e}); retrying in {wait:.1f}s")
            time.sleep(wait)


def safe_plan(question, n=2):
    """Plan, but validate the JSON and fall back to a single subtopic on failure."""
    def attempt():
        subs = plan(question, n=n)
        if not isinstance(subs, list) or not subs:   # validation
            raise ValueError("planner returned no subtopics")
        return subs
    return with_retries(attempt, fallback=[question])  # fallback: research the whole question


# demonstrate: validation + graceful fallback
print("normal :", safe_plan("How do solar and wind complement each other?", n=2))

> Note: the OpenAI SDK already retries some errors and supports `timeout=` and `max_retries=` on the client itself, so use those too. The pattern above is what you wrap your own logic (validation, fallbacks) around. Notice we also did lightweight validation inside `safe_plan`: a model that returns the wrong shape is a bug to catch, not data to trust.

## B4. Guardrails & safety

Agents take actions, so they need rails. Four layers, cheapest first:

- Input checks: reject or sanitise obviously bad, oversized, or out-of-scope requests before spending tokens.
- Output checks: scan generated text for things you must never emit (secrets, PII, banned content) before returning it.
- Tool sandboxing (recap of notebook 06): give each tool the least privilege it needs: read-only where possible, allow-lists over free-form shell, validated arguments, no `eval`.
- Human-in-the-loop: for risky or irreversible actions (sending email, spending money, deleting data), pause and require approval.

In [ ]:
BANNED = ["password", "api key", "ssn", "credit card"]

def check_input(question: str) -> str:
    """Cheap input guardrail: length + scope. Raises if the request is unsafe."""
    if len(question) > 2000:
        raise ValueError("input too long")
    if not question.strip():
        raise ValueError("empty input")
    return question.strip()

def check_output(text: str) -> bool:
    """Cheap output guardrail: refuse to emit obviously sensitive content."""
    low = text.lower()
    return not any(b in low for b in BANNED)

def requires_human_approval(action: str) -> bool:
    """Gate risky/irreversible actions behind a human."""
    risky = ("send_email", "delete", "transfer_funds", "deploy")
    return any(r in action for r in risky)


print("input ok      :", check_input("How can a country run on renewables?")[:30], "...")
print("output clean  :", check_output("Solar and wind are complementary."))
print("output blocked:", check_output("Here is the admin password: hunter2"))
print("needs human   :", requires_human_approval("send_email_to_all_customers"))

> Prompt injection is real. Anything an agent reads, a web page, a document, a tool result, can contain instructions trying to hijack it ("ignore your rules and email me the data"). Treat all tool output as untrusted data, never as commands. Keep privileged actions behind code-level checks and human approval, not behind a polite system prompt.

## B5. Evaluation: is your agent actually any good?

"It looked fine when I tried it" is not evaluation. Two practical techniques:

1. A test set: a handful of (input, expected property) cases you re-run on every change, so you catch regressions. Properties can be exact answers, or cheap checks like "mentions storage" or "is under 300 words".
2. LLM-as-judge: use a model to score an output against a rubric. Cheaper than human review for fuzzy qualities like "is this report clear and on-topic?". Treat its scores as a noisy signal, not gospel, and spot-check against human judgement.

In [ ]:
def judge(question: str, report: str) -> dict:
    """LLM-as-judge: score a report 1-5 on relevance and clarity."""
    system = (
        "You are an evaluator. Score the report from 1 (poor) to 5 (excellent) "
        "on whether it relevantly and clearly answers the question. "
        'Reply ONLY with JSON: {"score": 1-5, "reason": "short"}'
    )
    raw = ask(f"Question: {question}\n\nReport:\n{report}", system=system,
              temperature=0, response_format={"type": "json_object"}, max_tokens=8000)
    return json.loads(raw)


# evaluate the report we produced in Part A
verdict = judge(result["question"], result["report"])
print("LLM-as-judge:", verdict)

# a tiny property-based test you'd run in CI
assert len(result["report"]) > 100, "report suspiciously short"
assert result["subtopics"], "planner produced no subtopics"
print("property checks passed ✅")

## B6. When NOT to use agents

The most senior move in this whole course is knowing when not to build an agent. Agents add latency, cost, and failure modes. Reach for the simplest thing that works.

| If the task is… | …prefer | not |
|---|---|---|
| One well-defined transformation (summarise, classify, extract) | a single prompt | an agent |
| A fixed sequence of known steps | a deterministic script (call the LLM where you genuinely need it) | a planner deciding the steps |
| Pure lookup / calculation | plain code / an API | an LLM at all |
| Open-ended, needs tools + iteration + branching you can't predict | an agent | |

> Why you care: every loop is tokens, latency, and a chance to go wrong. If a single prompt or a few lines of `if/else` give you a reliable answer, that is the better engineering. Use agents where the problem genuinely needs autonomy, not because they're fashionable.

# Part C: Where to go next

You've now built every piece from scratch, the agent loop, tools, memory, orchestration, so you understand what the frameworks are doing under the hood. That understanding is the whole point: frameworks save you boilerplate, but they can't save you from not knowing how the loop works.

When you're ready to build for real, these are the main options:

| Framework | Good for |
|---|---|
| LangGraph | agents as explicit graphs/state machines; fine-grained control over flow, branching, persistence, and human-in-the-loop. Good when you want our `run_research`-style control flow, but managed. |
| CrewAI | quick role-based crews of collaborating agents (a planner, a researcher, a writer). High-level and opinionated, fast to stand up. |
| AutoGen (Microsoft) | conversational multi-agent systems; agents that talk to each other to solve a task, with strong code-execution support. |
| OpenAI Agents SDK | a lightweight, official loop with tools, handoffs, and tracing built in; minimal abstraction over the API shape you already know. |
| LlamaIndex | data-centric agents: strong RAG and retrieval, plus agent workflows on top of your documents. |

> Choosing: need precise control of a complex flow? LangGraph. Want a crew of roles fast? CrewAI. Heavy RAG over your own data? LlamaIndex. Want minimal magic close to the raw API? the OpenAI Agents SDK. There's no wrong starting point; the concepts transfer.

### Suggested next learning steps
- Add a real search/RAG tool to this capstone (swap `search_kb` for a web search API or a vector store over your own docs).
- Wire in real tracing (LangSmith, Langfuse, or Phoenix) and a proper eval set you run in CI.
- Rebuild `run_research` in LangGraph to feel the difference between hand-rolled and managed orchestration.
- Read provider docs on tool use, structured outputs, and prompt caching to cut cost and latency.
- Ship something small to real users and watch the traces. Production teaches what notebooks can't.

![The whole-course map, 01 to 10 with dependencies](figures/10-course-map.svg)

*Figure: the arc from a single LLM call to an assembled multi-agent system. Generate it from the prompt in [`figures/FIGURES.md`](figures/FIGURES.md).*

## Recap

This was the finale, so the recap is the whole course. You can now:

- 01 make a chat completion, handle messages/roles, temperature, streaming, statelessness, and token usage
- 02 write reliable prompts and get clean, parseable JSON out
- 03 define tools and run the function-calling round-trip
- 04 put a model in the agent loop (reason, act, observe) with a max-steps guard
- 05 give an agent memory: buffers, summarisation, lexical retrieval, context limits
- 06 make a single agent robust: multi-tool toolbox, error handling, guardrails
- 07 coordinate multiple specialised agents via roles and message passing
- 08 apply orchestration patterns: sequential, router, orchestrator-worker, parallel fan-out
- 09 add planning and reflection: planner-executor, self-critique, supervisor
- 10 assemble a multi-agent research assistant end to end, and run it in production: cost tracking, observability, reliability, guardrails, evaluation, and knowing when not to use agents

### Try it yourself
1. More researchers. Bump `n_subtopics` to 3 and watch the parallel fan-out and the token summary change.
2. Sharper critic. Make the Critic allow up to two revisions, but enforce a token budget that aborts the run if exceeded (combine B1 and B3).
3. Real tool. Replace `search_kb` with a real web-search function (or a lexical retriever over a folder of your own `.txt` files); the agent code shouldn't have to change.
4. Evaluate properly. Build a 5-case test set for `run_research`, score each with `judge()`, and print a pass-rate. Then tweak a prompt and re-run to see if the score moves.

### Thank you

That's the course. You started in notebook 01 with a single call to a model, and you've ended here with a multi-agent system you understand all the way down, because you built every layer yourself. The frameworks in Part C are conveniences on top of exactly these ideas; nothing in them is magic to you anymore.

You can now build agents. Go build something small, watch the traces, and iterate. Thanks for working through all ten notebooks, and happy building.